# 6. Concurrency & Parallelism — CRITICAL

This is one of the most important areas for a senior backend Python role.

Python concurrency is not one single model. It is a set of tools with different trade-offs:

- `threading` for shared-memory I/O work
- `multiprocessing` for CPU-heavy parallelism
- `asyncio` for massive concurrent I/O with cooperative scheduling

A strong backend engineer knows not only the syntax, but also when each model is appropriate.

---

## 1) Threading

`threading` lets you run multiple threads in the same process. Threads share memory, so they are efficient for I/O-bound work, but they are dangerous if shared state is modified without synchronization.

### Thread lifecycle

A thread typically goes through:

1. Create the thread object
2. Start the thread
3. Run the target function
4. Join the thread to wait for completion
5. Exit when the function ends

Important idea:
- Threads run concurrently, not necessarily in parallel.
- On CPython, the GIL limits CPU-level parallelism, but threads still help with I/O waiting.

### Thread pools

Thread pools reuse worker threads instead of creating a new thread for every task.
This reduces overhead and keeps resource use under control.

The main API is `ThreadPoolExecutor`.

### Key synchronization primitives

#### Lock

A `Lock` allows only one thread to enter a critical section at a time.

#### RLock

An `RLock` is a re-entrant lock. The same thread can acquire it multiple times.
It is useful when code paths may call lock-protected helper functions recursively.

#### Semaphore

A `Semaphore` limits how many threads can access a resource at once.
Useful for connection pools and rate-limiting patterns.

#### Event

An `Event` is a simple flag-like synchronization mechanism.
One thread waits on the event; another thread sets it.

#### Condition

A `Condition` is a lock + wait/notify mechanism.
Used when a thread must wait until a certain state is reached.

#### Barrier

A `Barrier` makes multiple threads wait until all of them reach the same point before continuing.
Useful for synchronized stages.

### Race conditions

A race condition happens when multiple threads access shared state without ordering guarantees.

Example:
- Two threads both read a counter value
- Both increment it
- Both write back
- Final value is lower than expected because the increments overlap

This is a classic bug in multithreaded code.

### Deadlock

A deadlock happens when threads are waiting on each other forever.

Example:
- Thread A holds Lock 1 and waits for Lock 2
- Thread B holds Lock 2 and waits for Lock 1

Neither can finish.

### Starvation

Starvation occurs when a thread is perpetually denied CPU time or access to a resource.
This often happens with unfair scheduling or bad lock design.

### Senior backend takeaway

Threading is useful when:
- the process is waiting on network calls, file I/O, or DB responses
- a task is mostly blocked, not computing

It is not the best choice for pure CPU-bound work in Python.

In [2]:
import threading
import time


def worker(name, delay):
    print(f"Thread {name} starting")
    time.sleep(delay)
    print(f"Thread {name} finishing")


threads = [
    threading.Thread(target=worker, args=("A", 1)),
    threading.Thread(target=worker, args=("B", 2)),
]

for t in threads:
    t.start()

for t in threads:
    t.join()

print("All threads completed")


Thread A starting
Thread B starting
Thread A finishing
Thread B finishing
All threads completed


## 2) Multiprocessing

`multiprocessing` creates separate processes, each with its own Python interpreter and memory space.
This is the right model for CPU-bound tasks because each process can run on a different core.

### Process vs thread

Thread:
- same process
- shared memory
- cheaper to create
- weaker isolation
- good for I/O-bound work

Process:
- separate memory space
- no shared memory by default
- more expensive
- better for CPU-heavy work
- can run truly in parallel on multi-core machines

### Process pools

Process pools are the multiprocessing equivalent of thread pools.
They are used to distribute many tasks across worker processes.

The main API is `ProcessPoolExecutor`.

### IPC

IPC means inter-process communication.
Processes must communicate explicitly because they do not share memory.

Common methods:
- `Queue`
- `Pipe`
- shared memory
- file-based communication

### Serialization

When data needs to move between processes, it must be serialized into a format that can be transmitted or stored.

### Pickling

Pickling converts Python objects into a byte stream for storage or transfer.
`pickle` is the standard library mechanism for Python object serialization.

Important points:
- not all objects are picklable
- pickling is required for process communication in many multiprocessing examples
- it is a common interview topic because debugging serialization issues is frequent in distributed and parallel systems

### Shared memory

Shared memory allows separate processes to access the same memory region.
This can improve performance for data-intensive parallel jobs, but it introduces synchronization complexity.

### `multiprocessing` in practice

A backend engineer should remember that:
- `threading` is best for I/O waiting
- `multiprocessing` is best for CPU-heavy work
- processes do not naturally share state, so if you need coordination you use queues, pipes, values, or shared memory

### Senior backend takeaway

Multiprocessing is best for:
- CPU-bound tasks
- image processing
- ML inference
- scientific workloads
- heavy data operations


In [6]:
from multiprocessing import Process
import os


def worker():
    print(f"Child process ID: {os.getpid()}")


if __name__ == "__main__":
    p = Process(target=worker)
    p.start()
    p.join()
    print(f"Parent process ID: {os.getpid()}")


Parent process ID: 6932


## 3) Async programming

`asyncio` is a single-threaded cooperative concurrency model built around an event loop.
It is designed for many concurrent I/O operations, especially network and database code.

### `async` and `await`

`async def` defines a coroutine.
`await` pauses the coroutine and yields control back to the event loop until the awaited work completes.

This is the core mechanic behind async Python.

### Coroutine

A coroutine is a function that can be paused and resumed.
It is not a thread and does not run in parallel by itself.
It cooperates with the event loop.

### Event loop

The event loop is the scheduler that runs ready coroutines and handles I/O completion callbacks.
It continuously cycles through tasks while waiting for operations to finish.

This is the heart of `asyncio`.

### Task

A `Task` is a scheduled coroutine running under the event loop.
It gives you a handle to a coroutine that is executing concurrently with other tasks.

### Future

A `Future` is a low-level object representing a result that may not be ready yet.
Tasks are built on top of futures.

### `asyncio`

`asyncio` provides the framework:
- event loop
- tasks
- futures
- queue primitives
- timeouts and cancellation
- synchronization primitives for async code

### `create_task`

`asyncio.create_task(...)` schedules a coroutine to run concurrently as a task.
This is the most common concurrency primitive in modern async Python.

### `gather`

`asyncio.gather(...)` runs multiple awaitables concurrently and waits for all of them to finish.
It returns their results in order.

This is often used when many independent I/O operations should happen simultaneously.

### Cancellation

A task can be cancelled with `task.cancel()`.
If a coroutine is awaiting something, cancellation propagates as `CancelledError`.

This matters in long-running network or worker tasks.

### Timeouts

`asyncio.wait_for(...)` wraps an operation and raises a timeout if it takes too long.
This is essential in network-heavy services to avoid hanging requests.

### Async locks

Async locks are the async equivalent of thread locks.
They prevent multiple coroutines from entering a critical section at once.

### Async queues

Async queues are used for producer-consumer patterns in async programs.
They let code push values into a queue and consume them safely.

### Async context managers

Async context managers support `async with` patterns.
These are used for async resources like locks, sessions, and database connections.

### Why `asyncio` matters in backend work

`asyncio` is specifically designed around concurrent asynchronous code and is heavily used underneath high-performance network and database frameworks.

It is especially valuable when:
- many requests are in flight
- network calls dominate waiting time
- database drivers and HTTP frameworks are async-first

### Senior backend takeaway

`asyncio` is the right model when the workload is mostly waiting on I/O and you want many lightweight concurrent operations without creating a thread per task.


In [8]:
import asyncio


async def worker(name, delay):
    print(f"Task {name} started")
    await asyncio.sleep(delay)
    print(f"Task {name} finished")
    return f"{name} done"


async def main():
    tasks = [
        asyncio.create_task(worker("A", 1)),
        asyncio.create_task(worker("B", 2)),
        asyncio.create_task(worker("C", 0.5)),
    ]

    results = await asyncio.gather(*tasks)
    print(results)


await main()

Task A started
Task B started
Task C started
Task C finished
Task A finished
Task B finished
['A done', 'B done', 'C done']


## 4) Most important comparison

### When would you choose threading vs multiprocessing vs asyncio?

#### Choose threading when:
- the workload is I/O-bound
- you have blocking file or network operations
- you want lower complexity than full async code
- the work is not pure CPU computation

#### Choose multiprocessing when:
- the workload is CPU-bound
- you want true parallelism across cores
- you are doing ML, image processing, scientific computing, or heavy number crunching
- native extensions or Cython code are being used

#### Choose asyncio when:
- you have many concurrent I/O tasks
- you need high throughput with low overhead
- the application is naturally event-driven
- you are working with HTTP clients, async web servers, or async database drivers

### Quick decision guide

- CPU-heavy -> multiprocessing / native extensions
- I/O-heavy synchronous -> threading
- Massive concurrent I/O -> asyncio
- Distributed work -> queue / workers
- External API calls -> asyncio / threads
- CPU-heavy ML operation -> process / native implementation

---

## 5) Deep interview answer

A senior Python backend engineer usually answers this way:

> If a task is mostly waiting for I/O, threading or asyncio is appropriate. If the task is doing real computation, use multiprocessing. The deciding factor is whether the workload is blocked by external systems or saturated by CPU work.

A more refined answer:

> Threading is good for a few blocking operations in a shared-memory process. Asyncio is better when you need thousands of concurrent I/O tasks without the overhead of one thread per task. Multiprocessing is the right tool when you need CPU parallelism across cores.

---

## 6) Final senior-level notes

A strong Python backend engineer should think in terms of workload shape, not just syntax.

- If the code is waiting on I/O, concurrency can help a lot.
- If the code is doing CPU work, native parallelism is required.
- Threading is not a substitute for multiprocessing in CPU-heavy tasks.
- Asyncio is a cooperative model, not a replacement for true parallelism.

The real skill is picking the right primitive for the real bottleneck.